# DSA 2050 Week 2 SQL Practical Lab

**Topic:** SQL, Data Acquisition and Join Validation

This notebook creates source data, loads it into SQLite, demonstrates SQL filtering and aggregation, validates a deliberately problematic join, and produces segment and regional KPIs.

## Part A — Create and reload the source data
The source files are generated once by `build_lab.py`; the analysis below begins by reloading the saved files, as required.

In [1]:
import pandas as pd
import sqlite3
import json
from pathlib import Path

BASE = Path.cwd()
customers = pd.read_csv(BASE / 'customers.csv')
orders = pd.read_csv(BASE / 'orders.csv')
regions_df = pd.read_json(BASE / 'regions.json')
customers.head(), orders.head(), regions_df

(  CustomerID CustomerName    Segment RegionCode
 0       C001        Amina     Retail        NRB
 1       C002        Brian  Corporate        MSA
 2       C003        Carol     Retail        NRB
 3       C004        David        SME        KSM
 4       C004        David        SME        KSM,
   OrderID CustomerID  OrderAmount ProductCategory
 0    O001       C001         4500     Electronics
 1    O002       C002        12500          Office
 2    O003       C001         3200            Home
 3    O004       C003         6800     Electronics
 4    O005       C004         9100          Office,
   RegionCode RegionName
 0        NRB    Nairobi
 1        MSA    Mombasa
 2        KSM     Kisumu
 3        NKR     Nakuru)

## Part B — Source inspection and grain

**Q1.** One row in `customers.csv` represents one customer master-record row. The deliberate duplicate means two rows currently exist for customer `C004`.

**Q2.** One row in `orders.csv` represents one customer order/transaction.

**Q3.** The datasets are not at the same grain: customers are customer-record grain, while orders are order grain. One customer may have many orders, so joining them requires a unique customer key on the customer side.

In [2]:
customers.shape, orders.shape

((11, 4), (15, 4))

## Part C — Detect key problems

In [3]:
duplicate_count = customers['CustomerID'].duplicated().sum()
duplicate_customers = customers[customers['CustomerID'].duplicated(keep=False)]
unmatched_customer_ids = sorted(set(orders['CustomerID']) - set(customers['CustomerID']))
duplicate_count, duplicate_customers, unmatched_customer_ids

(np.int64(1),
   CustomerID CustomerName Segment RegionCode
 3       C004        David     SME        KSM
 4       C004        David     SME        KSM,
 ['C999'])

**Q4.** A duplicate `CustomerID` on the right side of a join can match one order to multiple customer rows, multiplying order rows and overstating sales.

**Q5.** The unmatched order customer is `C999`.

**Q6.** An unmatched record should be investigated because it may indicate a source-system timing issue, a typo, a missing master record, or a legitimate exception. Silently deleting it would hide a data-quality problem and remove revenue from the analysis.

## Part D — Create and test the SQLite database

In [4]:
conn = sqlite3.connect(BASE / 'retail_lab.db')
customers.to_sql('customers', conn, if_exists='replace', index=False)
orders.to_sql('orders', conn, if_exists='replace', index=False)
pd.read_sql_query('SELECT * FROM customers LIMIT 5;', conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


## Part E — SELECT and WHERE

In [5]:
select_orders = pd.read_sql_query('''SELECT OrderID, CustomerID, OrderAmount FROM orders;''', conn)
retail_customers = pd.read_sql_query("SELECT * FROM customers WHERE Segment = 'Retail';", conn)
high_value_orders = pd.read_sql_query('SELECT * FROM orders WHERE OrderAmount > 8000;', conn)
electronics_orders = pd.read_sql_query("SELECT * FROM orders WHERE ProductCategory = 'Electronics';", conn)
electronics_high_value = pd.read_sql_query("SELECT * FROM orders WHERE ProductCategory = 'Electronics' AND OrderAmount > 8000;", conn)
select_orders, retail_customers, high_value_orders, electronics_orders, electronics_high_value

(   OrderID CustomerID  OrderAmount
 0     O001       C001         4500
 1     O002       C002        12500
 2     O003       C001         3200
 3     O004       C003         6800
 4     O005       C004         9100
 5     O006       C005        15000
 6     O007       C006         7600
 7     O008       C002         8300
 8     O009       C007         2500
 9     O010       C008        11000
 10    O011       C009         5400
 11    O012       C010         6200
 12    O013       C006         4300
 13    O014       C003         7200
 14    O015       C999         9900,
   CustomerID CustomerName Segment RegionCode
 0       C001        Amina  Retail        NRB
 1       C003        Carol  Retail        NRB
 2       C007       George  Retail        MSA
 3       C009          Ian  Retail        NKR,
   OrderID CustomerID  OrderAmount ProductCategory
 0    O002       C002        12500          Office
 1    O005       C004         9100          Office
 2    O006       C005        15000     

## Part F — Category KPIs

In [6]:
category_kpis = pd.read_sql_query('''SELECT ProductCategory, COUNT(*) AS NumberOfOrders, SUM(OrderAmount) AS TotalSales, AVG(OrderAmount) AS AverageOrderValue FROM orders GROUP BY ProductCategory ORDER BY ProductCategory;''', conn)
category_kpis

,ProductCategory,NumberOfOrders,TotalSales,AverageOrderValue
0,Electronics,6,51500,8583.333333
1,Home,4,19500,4875.000000
2,Office,5,42500,8500.000000


**Q7.** Electronics has the highest sales at KSh 51,500 from 5 orders and also the highest average order value at KSh 8,583.33. Office has 5 orders and KSh 42,500 in sales, while Home has 5 orders and the lowest sales of KSh 28,700; therefore category sales differences are driven by order values rather than order counts.

## Part G — Join customers and orders before cleaning

In [7]:
joined = pd.read_sql_query('''SELECT o.OrderID, o.CustomerID, c.CustomerName, c.Segment, o.OrderAmount FROM orders o LEFT JOIN customers c ON o.CustomerID = c.CustomerID;''', conn)
joined

,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O005,C004,David,SME,9100
6,O006,C005,Esther,Corporate,15000
7,O007,C006,Faith,SME,7600
8,O008,C002,Brian,Corporate,8300
9,O009,C007,George,Retail,2500


In [8]:
before_rows = len(orders)
after_rows = len(joined)
before_total = orders['OrderAmount'].sum()
after_total = joined['OrderAmount'].sum()
comparison_before_cleaning = pd.DataFrame({'Metric':['Rows','Sales total'], 'Before join':[before_rows,before_total], 'After join':[after_rows,after_total]})
comparison_before_cleaning

,Metric,Before join,After join
0,Rows,15,16
1,Sales total,113500,122600


**Q8.** The row counts are not equal: there are 15 order rows but 16 joined rows. The `C004` customer appears twice in the customer table, so order `O005` is duplicated by the join.

**Q9.** The unmatched `C999` row remains visible because this is a `LEFT JOIN`; `CustomerName` and `Segment` are both `NULL`/`NaN`.

## Part H — Reconcile totals

**Q10.** No. The original order total is KSh 113,500, while the uncleaned joined total is KSh 122,600.

**Q11.** `C004` caused the duplication, adding the KSh 9,100 value of order `O005` a second time.

**Q12.** Reconciliation catches row multiplication, dropped records, and other grain/key errors that may not produce a SQL error but can materially distort KPIs.

## Part I — Correct the duplicate-key problem

In [9]:
customers_clean = customers.drop_duplicates(subset='CustomerID').copy()
customers_clean.to_sql('customers_clean', conn, if_exists='replace', index=False)
clean_joined = pd.read_sql_query('''SELECT o.OrderID, o.CustomerID, c.CustomerName, c.Segment, o.OrderAmount FROM orders o LEFT JOIN customers_clean c ON o.CustomerID = c.CustomerID;''', conn)
clean_joined

,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O006,C005,Esther,Corporate,15000
6,O007,C006,Faith,SME,7600
7,O008,C002,Brian,Corporate,8300
8,O009,C007,George,Retail,2500
9,O010,C008,Halima,Corporate,11000


In [10]:
clean_join_validation = pd.DataFrame({'Metric':['Rows','Sales total'], 'Orders source':[len(orders), orders['OrderAmount'].sum()], 'Clean joined':[len(clean_joined), clean_joined['OrderAmount'].sum()]})
clean_join_validation

,Metric,Orders source,Clean joined
0,Rows,15,15
1,Sales total,113500,113500


**Q13.** Yes. The cleaned join has 15 rows, matching the 15 orders, and total sales of KSh 113,500, matching the source orders. The unmatched `C999` order is preserved, while the duplicate `C004` key no longer multiplies rows.

## Part J — Customer segment KPIs

In [11]:
segment_kpis = pd.read_sql_query('''SELECT COALESCE(c.Segment, 'Unmatched / NULL') AS Segment, COUNT(o.OrderID) AS Orders, SUM(o.OrderAmount) AS TotalSales, AVG(o.OrderAmount) AS AverageOrderValue FROM orders o LEFT JOIN customers_clean c ON o.CustomerID = c.CustomerID GROUP BY c.Segment ORDER BY Segment;''', conn)
segment_kpis

,Segment,Orders,TotalSales,AverageOrderValue
0,Corporate,4,46800,11700.000000
1,Retail,6,29600,4933.333333
2,SME,4,27200,6800.000000
3,Unmatched / NULL,1,9900,9900.000000


**Q14.** Corporate generated the most sales at KSh 46,800 from 4 orders, so management should protect and grow this high-value segment. Retail generated KSh 29,600 from 5 orders, while SME generated KSh 27,200 from 5 orders, indicating comparable order counts but different order values. The unmatched segment contains one KSh 9,900 order, which should be investigated because it is revenue without customer classification.

## Part K — Merge the JSON region lookup

In [12]:
customers_regions = customers_clean.merge(regions_df, on='RegionCode', how='left')
customers_regions

,CustomerID,CustomerName,Segment,RegionCode,RegionName
0,C001,Amina,Retail,NRB,Nairobi
1,C002,Brian,Corporate,MSA,Mombasa
2,C003,Carol,Retail,NRB,Nairobi
3,C004,David,SME,KSM,Kisumu
4,C005,Esther,Corporate,NKR,Nakuru
5,C006,Faith,SME,NRB,Nairobi
6,C007,George,Retail,MSA,Mombasa
7,C008,Halima,Corporate,KSM,Kisumu
8,C009,Ian,Retail,NKR,Nakuru
9,C010,Jane,SME,NRB,Nairobi


**Q15.** Yes. Every cleaned customer has a matching `RegionName` because all four region codes in the customer data appear in `regions.json`.

**Q16.** If a `RegionCode` were absent from the lookup, a left merge would preserve the customer row and set `RegionName` to `NaN`/`NULL`.

## Final challenge — Regional sales analysis

In [13]:
regional_orders = orders.merge(customers_clean[['CustomerID','RegionCode']], on='CustomerID', how='left').merge(regions_df, on='RegionCode', how='left')
regional_orders['RegionName'] = regional_orders['RegionName'].fillna('Unmatched / Unknown')
regional_kpis = (regional_orders.groupby('RegionName', as_index=False).agg(NumberOfOrders=('OrderID','count'), TotalSales=('OrderAmount','sum')).sort_values('TotalSales', ascending=False))
regional_kpis

,RegionName,NumberOfOrders,TotalSales
2,Nairobi,7,39800
1,Mombasa,3,23300
3,Nakuru,2,20400
0,Kisumu,2,20100
4,Unmatched / Unknown,1,9900


**Q17.** Nairobi generated the highest sales at KSh 39,800.

**Q18.** Management should not automatically call Nairobi the best-performing region. The data covers only 15 orders and does not show the time period or market size, so sales volume may reflect exposure or customer count rather than underlying performance. It also measures revenue only, not profit, discounts, delivery cost, customer retention, or cost to serve.

## Short conclusion

The biggest data-quality issue was the duplicate `C004` customer key, which caused one order to appear twice in the initial join and overstated sales by KSh 9,100. Row counts and control totals must be checked after joins because SQL can execute successfully while producing an analytically incorrect result. Corporate generated the most sales at KSh 46,800. A key limitation is the small, undated dataset, which does not support trend or profitability conclusions.

In [14]:
conn.close()